# FDTD — small allocation case

Generate real solver inputs with gsim, measure their sizing requirements, and send those requirements directly to the cloud API before upload and submission. The service owns allocation policy; the category in this filename is an expected test outcome, not an allocation instruction.

Run from this directory with the gsim environment selected as the notebook kernel. Set `SIMULATION_API_URL` to the simulation API base ending in `/v1`, and set `GFP_API_KEY` in the kernel environment. Neither value belongs in saved outputs.

The final cell creates and starts a real cloud job. It prints the job ID and requested resources without waiting for completion.

In [ ]:
CASE_NAME = "fdtd-small"
GRID_CELLS = (500, 500, 200)

In [ ]:
from pathlib import Path

import gdsfactory as gf
from helpers import submit_case, write_request

case_dir = Path.cwd() / "output" / CASE_NAME
input_dir = case_dir / "inputs"
input_dir.mkdir(parents=True, exist_ok=True)

## Configure a straight waveguide

The domain dimensions and Yee spacing target the requested cell count, including PML. A coarse tetrahedral geometry mesh keeps local input generation small; it does not reduce the Yee-grid allocation. These settings test scheduling and launch, not optical accuracy.

In [ ]:
from helpers import fdtd_requirements, write_fdtd_grid_bounds

from gsim import fdtd

gf.gpdk.PDK.activate()
cell_size_nm = 50.0
pml_cells = 12
extent_um = [(count - 2 * pml_cells) * cell_size_nm / 1000 for count in GRID_CELLS]
length_um, width_um, height_um = extent_um
component = gf.components.straight(length=length_um, cross_section="strip")

simulation = fdtd.Simulation()
simulation.materials(background="SiO2", overrides={"si": 3.4757})
# This is the tetrahedral geometry mesh size, independent of the Yee cell size.
simulation.geometry(component, mesh_size_nm=5000.0)
simulation.source(
    port="o1", wavelength_um=1.55, wavelength_span_um=0.1, num_wavelengths=3
)
simulation.domain(
    x_bounds=(0.0, length_um),
    y_bounds=(-width_um / 2, width_um / 2),
    z_bounds=(0.11 - height_um / 2, 0.11 + height_um / 2),
    pml_cells=pml_cells,
)
# Short execution tests allocation and launch, rather than converged S-parameters.
simulation.solver(cell_size_nm=cell_size_nm, max_timesteps=10, max_wall_seconds=60)
print({"target_grid_cells": GRID_CELLS, "domain_um": extent_um})

## Generate and measure inputs

The measured count uses the written mesh bounds and config spacing, following gsim’s Yee-grid display convention: `ceil(extent / spacing) + 2 * PML` on each axis. It is a sizing measurement; the solver may adjust its final grid. No allocation thresholds or machine sizes are calculated here.

In [ ]:
artifacts = simulation.write(input_dir)
write_fdtd_grid_bounds(artifacts.mesh_path, artifacts.config_path)
requirements = fdtd_requirements(artifacts.mesh_path, artifacts.config_path)
input_files = [artifacts.mesh_path, artifacts.config_path]
request = write_request(case_dir, CASE_NAME, requirements)

## Send sizing JSON, upload inputs, and submit

`helpers.submit_case` calls `/job/prepare` with `sizing_requirements`, uploads the input ZIP to the returned presigned URL, then calls `/job/submit`. This explicit notebook adapter avoids older SDK uploads that omit the new field. The target server must support sizing requirements and have sizing profiles enabled.

`metadata.json` and `prepare.json` stay beside `source.zip`; the ZIP contains only the mesh and solver config. This adapter sends metadata directly to the API. Uploading a separate metadata sidecar to S3 still requires SDK/backend support for a sidecar upload URL.

Supported cases print requested CPU and RAM from the server response. Confirm these in the cloud job/admin view against the service’s current profiles. The filename alone does not prove a tier was selected. Failed HTTP requests raise errors; no fallback to fixed-size SDK submission occurs.

In [ ]:
cloud_result = submit_case(
    case_dir,
    request,
    input_files,
    expect_rejection=False,
)
cloud_result